# 跨云平台 Checkpoint 导出 / 导入工具

用于在云平台 A 中止训练后，把**训练成果**（checkpoint + 字体 + charsets）打包带走，
在新平台 B 上**接着训练**，而无需搬运庞大的 `data/` 图像数据集
（`data/` 可由 `fonts/` + `charsets/` + 固定随机种子在新平台重建）。

## 重要前提
- 新旧平台 `fonts/` 下的目标字体**文件名必须一致**（checkpoint 路径与 charset 子目录都按字体名派生）。
- 新平台仍需按原流程跑「数据准备 Cell」重建 `data/`，再 resume 本 notebook 导出的 checkpoint。
- 代码本身（本项目）通过 git clone / 上传获取，**本 notebook 不打包代码**。

In [8]:
# ============================================================
# Cell 1【导出端 / 平台 A】：检测 checkpoints 并打包为 zip
# ============================================================
# 需要打包的目录：
#   checkpoints/  - 训练状态（模型+优化器+调度器+epoch）
#   fonts/        - 目标字体 + 参考字体（决定渲染结果）
#   charsets/     - 训练字符集（决定训练哪些字）
# 无需打包：data/（可重建）、项目代码（git 获取）

import os
import zipfile
from pathlib import Path

start_dir = Path.cwd()  # notebook 原始工作目录（腾讯云 UI 可见，zip 将输出到这里）

# ============ 自动定位项目根目录 ============
# 训练 notebook（Cell 0）会 os.chdir 到项目根目录；但本 notebook 单独运行
# 时工作目录不一定是项目根目录（例如在 /workspace 下打开），
# 因此复用同样的搜索逻辑：先看当前目录，再在 /workspace、/home、/ 下找。
def _find_project_root():
    if os.path.isdir("scripts") and os.path.exists("requirements.txt"):
        return Path.cwd()
    for search_root in [os.getcwd(), "/workspace", "/home", "/"]:
        if not os.path.isdir(search_root):
            continue
        for entry in sorted(os.listdir(search_root)):
            cand = os.path.join(search_root, entry)
            if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "scripts")):
                return Path(cand)
    return None

_proj = _find_project_root()
if _proj is None:
    raise SystemExit("未找到项目根目录（含 scripts/），请手动设置 PROJECT_ROOT 后重跑。")
os.chdir(_proj)
print(f"已切换到项目根目录: {os.getcwd()}")
# ===========================================

# ---- 可调整：要打包的目录与输出文件名 ----
DIRS_TO_PACK = ["checkpoints", "fonts", "charsets"]
OUTPUT_ZIP = start_dir / "hanzi_transfer.zip"   # 输出到 start_dir（如 /workspace，腾讯云 UI 可直接看到下载）
# -----------------------------------------

missing = [d for d in DIRS_TO_PACK if not Path(d).exists()]
if missing:
    print("⚠️ 以下目录不存在，将被跳过：", missing)

present = [d for d in DIRS_TO_PACK if Path(d).exists()]

# 检查 checkpoints 是否真的有内容（避免空目录被误当已训练）
ckpt_dir = Path("checkpoints")
ckpt_files = list(ckpt_dir.glob("*.pth")) if ckpt_dir.exists() else []
if ckpt_dir in map(Path, present) and not ckpt_files:
    print("⚠️ checkpoints/ 目录存在但没有 .pth 文件，尚未产生任何 checkpoint！")

if not present:
    raise SystemExit("没有任何可打包目录，导出终止。")

# 若已存在同名 zip 先删除
if Path(OUTPUT_ZIP).exists():
    Path(OUTPUT_ZIP).unlink()

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in present:
        for root, _, files in os.walk(d):
            for f in files:
                fp = Path(root) / f
                zf.write(fp, fp)  # 用相对路径归档

size_mb = Path(OUTPUT_ZIP).stat().st_size / 1024 / 1024
print(f"✅ 已导出: {Path(OUTPUT_ZIP).resolve()}  ({size_mb:.1f} MB)")
print("   包含目录:", present)
print(f"   checkpoint 文件数: {len(ckpt_files)}")
print("\n📦 请将此 zip 下载到本地，再上传到新平台后运行 Cell 2。")

已切换到项目根目录: /HanziGen_ICWfork
✅ 已导出: /HanziGen_ICWfork/hanzi_transfer.zip  (55.5 MB)
   包含目录: ['checkpoints', 'fonts', 'charsets']
   checkpoint 文件数: 1

📦 请将此 zip 下载到本地，再上传到新平台后运行 Cell 2。


In [ ]:
# ============================================================
# Cell 2【导入端 / 平台 B】：解压到指定位置
# ============================================================
# 注意：解压前请确认新平台已通过 git 拿到本项目代码框架，
# 且 fonts/ 下的目标字体文件名与平台 A 完全一致。

import os
from pathlib import Path
import zipfile

start_dir = Path.cwd()  # notebook 原始工作目录（zip 通常上传到这里）

# ============ 自动定位项目根目录 ============
# 与 Cell 1 相同：确保解压到真正的项目根目录，而非 notebook 工作目录。
def _find_project_root():
    if os.path.isdir("scripts") and os.path.exists("requirements.txt"):
        return Path.cwd()
    for search_root in [os.getcwd(), "/workspace", "/home", "/"]:
        if not os.path.isdir(search_root):
            continue
        for entry in sorted(os.listdir(search_root)):
            cand = os.path.join(search_root, entry)
            if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "scripts")):
                return Path(cand)
    return None

_proj = _find_project_root()
if _proj is None:
    raise SystemExit("未找到项目根目录（含 scripts/），请先在新平台克隆项目代码。")
os.chdir(_proj)
print(f"已切换到项目根目录: {os.getcwd()}")
# ===========================================

# ---- 可调整：zip 路径与解压根目录 ----
ZIP_PATH = "hanzi_transfer.zip"   # 上传后的 zip 位置（相对或绝对路径）
EXTRACT_ROOT = "."                 # 解压到项目根目录（目录结构会自动还原）
# -------------------------------------

# 相对路径按原始工作目录（如 /workspace）解析，避免 chdir 后指向项目根目录
if not os.path.isabs(ZIP_PATH):
    ZIP_PATH = str(start_dir / ZIP_PATH)

if not Path(ZIP_PATH).exists():
    raise SystemExit(f"找不到 {ZIP_PATH}，请先上传 zip 到该路径。")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    zf.extractall(EXTRACT_ROOT)

print(f"✅ 已解压 {len(names)} 个文件到 {EXTRACT_ROOT}")
top_dirs = sorted({n.split('/')[0] for n in names if '/' in n})
print("   还原的顶层目录:", top_dirs)

# 校验关键点
ckpt = list(Path("checkpoints").glob("*.pth")) if Path("checkpoints").exists() else []
print(f"\n📌 checkpoints/*.pth 数量: {len(ckpt)}")
for p in ckpt:
    print("   -", p.name)
print("\n👉 下一步：按原流程跑「数据准备 Cell」重建 data/（字体名需一致），")
print("   再运行训练 Cell；Cell 1 会自动检测到 checkpoints 并 resume 继续训练。")